# 00 - Definisanje problema

## Telco Customer Churn: predviđanje odliva korisnika

Pre nego što uopšte krenemo sa podacima, treba da bude jasno šta tačno predviđamo, zašto je to bitno i kako ćemo meriti da li model radi dobro.

Ovde nema obrade podataka — čisto uvodni sadržaj.

## 1. Kontekst
Telekomunikacione kompanije posluju na jako konkurentnom tržištu gde korisnici bez većih problema prelaze sa jednog provajdera na drugi. Kada korisnik napusti kompaniju, to se naziva odlivom korisnika (customer churn).

Zašto je to problem? Uglavnom zbog troškova — zadržati postojećeg korisnika je dosta jeftinije nego pridobiti novog. Zato se kompaniji više isplati da na vreme prepozna korisnike koji su u riziku od odlaska i pokuša da ih zadrži (popustom, boljom ponudom, pozivom podrške) nego da stalno juri nove.

Da bi to bilo moguće, kompanija mora unapred da zna ko će verovatno otići. To je i cilj ovog projekta: iz podataka o korisniku predvideti da li će napustiti kompaniju, kako bi se reagovalo na vreme.

## 2. Cilj našeg rada

Cilj je da napravimo model koji na osnovu podataka o korisniku predviđa vrednost kolone *Churn* — da li je korisnik otišao (*Yes*) ili ostao (*No*).

U terminima mašinskog učenja, ovo spada u nadgledano učenje (supervised learning), pošto za svakog korisnika iz skupa već znamo tačan ishod, pa model uči iz označenih primera. Konkretno, radi se o binarnoj klasifikaciji jer ciljna promenljiva ima samo dve moguće vrednosti (*Yes* / *No*).

Dakle, model ne predviđa broj (to bi bila regresija) nego kategoriju — svrstava korisnika u jednu od dve klase.

## 3. Opis skupa podataka

Telco Customer Churn skup sadrži podatke o 7.043 korisnika, raspoređene kroz 21 kolonu. Za svakog korisnika imamo osnovne podatke o njemu, koje usluge koristi, kakav ugovor ima, koliko i kako plaća i, ono što nas najviše zanima, da li je otišao iz kompanije.

Kolone se grubo mogu podeliti u četiri grupe:

**A. Demografija** — podaci o samom korisniku:
gender, SeniorCitizen, Partner, Dependents

**B. Usluge** — šta korisnik koristi:
PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies

**C. Nalog i plaćanje** — odnos korisnika sa kompanijom:
tenure, Contract, PaperlessBilling, PaymentMethod, MonthlyCharges, TotalCharges

**D. Ciljna promenljiva:**
`Churn` — da li je korisnik napustio kompaniju u prethodnom mesecu.

### Detaljan opis kolona

| Kolona | Opis | Tip |
|---|---|---|
| `customerID` | Jedinstveni ID korisnika. Nema veze sa churn-om pa ga izbacujemo. | Identifikator |
| `gender` | Pol: `Female`, `Male`. | Kategorijska (nominalna) |
| `SeniorCitizen` | Da li je stariji građanin: `0` = ne, `1` = da. Zapisano kao broj, ali je zapravo kategorija. | Kategorijska (binarna) |
| `Partner` | Da li korisnik ima partnera: `Yes`, `No`. | Kategorijska (binarna) |
| `Dependents` | Da li izdržava nekoga (npr. decu): `Yes`, `No`. | Kategorijska (binarna) |
| `tenure` | Koliko meseci je korisnik u kompaniji. | Numerička (diskretna) |
| `PhoneService` | Da li ima telefonsku uslugu: `Yes`, `No`. | Kategorijska (binarna) |
| `MultipleLines` | Da li ima više linija: `Yes`, `No`, `No phone service`. | Kategorijska (nominalna) |
| `InternetService` | Tip interneta: `DSL`, `Fiber optic`, `No`. | Kategorijska (nominalna) |
| `OnlineSecurity` | Online zaštita: `Yes`, `No`, `No internet service`. | Kategorijska (nominalna) |
| `OnlineBackup` | Online backup: `Yes`, `No`, `No internet service`. | Kategorijska (nominalna) |
| `DeviceProtection` | Zaštita uređaja: `Yes`, `No`, `No internet service`. | Kategorijska (nominalna) |
| `TechSupport` | Tehnička podrška: `Yes`, `No`, `No internet service`. | Kategorijska (nominalna) |
| `StreamingTV` | Streaming TV: `Yes`, `No`, `No internet service`. | Kategorijska (nominalna) |
| `StreamingMovies` | Streaming filmova: `Yes`, `No`, `No internet service`. | Kategorijska (nominalna) |
| `Contract` | Tip ugovora: `Month-to-month`, `One year`, `Two year`. | Kategorijska (ordinalna) |
| `PaperlessBilling` | Bez papirnog računa: `Yes`, `No`. | Kategorijska (binarna) |
| `PaymentMethod` | Način plaćanja: `Electronic check`, `Mailed check`, `Bank transfer (automatic)`, `Credit card (automatic)`. | Kategorijska (nominalna) |
| `MonthlyCharges` | Mesečni iznos naplate. | Numerička (kontinualna) |
| `TotalCharges` | Ukupno naplaćeno do sada. | Numerička (kontinualna) |
| `Churn` | Ciljna promenljiva — da li je korisnik otišao: `Yes`, `No`. | Kategorijska (binarna) |

## 4. Izazovi u podacima

Već iz prvog pregleda skupa mogu se naslutiti neki problemi na koje ćemo morati da obratimo pažnju:

**Skrivene nedostajuće vrednosti u `TotalCharges`.** Ova kolona je učitana kao tekst iako bi trebalo da bude broj. Ispostavlja se da 11 korisnika umesto vrednosti ima prazan razmak. Problem je što ih **isnull()** ne prepoznaje kao nedostajuće, pa ćemo morati ručno da ih uhvatimo i konvertujemo kolonu u numerički tip.

**Nebalansirane klase.** Korisnika koji su ostali (`Churn = No`) ima oko 73%, a onih koji su otišli (`Churn = Yes`) oko 27%. Zbog ovog nebalansa obična tačnost (accuracy) nije baš pouzdana metrika, pa ćemo verovatno morati da posegnemo za balansiranjem klasa i prikladnijim metrikama.

**Vrednosti `"No internet service"` i `"No phone service"`.** Dosta kolona (npr. `OnlineSecurity`, `StreamingTV`) ima i treću vrednost tipa `No internet service`, i tu treba biti oprezan jer to nije isto što i `No`:
- `No` — korisnik ima internet, ali ne koristi tu dodatnu uslugu.
- `No internet service` — korisnik uopšte nema internet.

Kako ćemo ovo tretirati odlucicemo kasnije, u fazi pripreme podataka.

Pošto su klase nebalansirane (~73% naspram ~27%), tačnost (accuracy) nam ovde ne znači mnogo. Model koji bi bukvalno uvek predviđao `No` imao bi oko 73% tačnosti, a bio bi potpuno beskoristan — ne bi uhvatio nijednog korisnika koji stvarno odlazi.

Zato ćemo model gledati kroz sledeće metrike:

- **Precision** — od svih koje je model označio kao „otići će", koliko je zaista otišlo.
- **Recall** — od svih koji su stvarno otišli, koliko ih je model uhvatio. Govori koliko odlazaka propuštamo.
- **F1-score** — harmonijska sredina precision-a i recall-a, kad hoćemo jedan broj koji balansira ta dva.
- **AUC-ROC** — koliko dobro model razdvaja dve klase, nezavisno od praga odlučivanja.

Nama je posebno bitan recall za klasu `Yes`. Iz poslovnog ugla, propustiti korisnika koji će otići (pa ne reagovati na vreme) obično je skuplje nego pogrešno označiti lojalnog korisnika kao rizičnog. Na to ćemo obraćati pažnju pri evaluaciji.

Sve metrike računamo samo na test skupu, koji ne diramo ni u pripremi podataka ni u treniranju.

In [2]:
# Uvozimo biblioteku pandas koja služi za rad sa tabelarnim podacima.
# Skraćenica "pd" je standardna konvencija — tako se pandas koristi svuda.
import pandas as pd


df = pd.read_csv("../WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Prikazujemo dimenzije skupa: df.shape vraća (broj_redova, broj_kolona).
# [0] uzima prvi element (redove), [1] drugi (kolone).
print("Broj redova:", df.shape[0])
print("Broj kolona:", df.shape[1])

# df.head() prikazuje prvih 5 redova tabele da vidimo kako podaci izgledaju.
# Kada je poslednja linija u ćeliji, Jupyter je automatski lepo prikaže.
df.head()

Broj redova: 7043
Broj kolona: 21


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
